<a href="https://colab.research.google.com/github/NiyatiPaghda/brain-tumor-mri-study/blob/main/dataset_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

zip_path = '/content/drive/MyDrive/Independent Study/3. Brain Tumor Project/data/raw/1512427.zip'

if os.path.exists(zip_path):
    print("Dataset found!")
    print(f"FIle size: {os.path.getsize(zip_path) / (1024**2):.2f} MB")
else:
    print("Dataset not found")

Dataset found!
FIle size: 838.76 MB


In [5]:
import zipfile

with zipfile.ZipFile(zip_path, 'r') as archive:
    files = archive.namelist()

    print("Total files:", len(files))
    print("\nFIrst 30 items:\n")

    for file in files[:30]:
        print(file)

Total files: 6

FIrst 30 items:

brainTumorDataPublic_1-766.zip
brainTumorDataPublic_1533-2298.zip
brainTumorDataPublic_767-1532.zip
brainTumorDataPublic_2299-3064.zip
cvind.mat
README 2024.txt


In [9]:
from collections import Counter

extension = Counter(
      os.path.splitext(file)[1].lower()
      for file in files
      if not file.endswith('/')
)

print("File types:")
for extension, count in extension.items():
    print(f"{extension or 'No extension'}: {count}")

File types:
.zip: 4
.mat: 1
.txt: 1


In [10]:
import zipfile

with zipfile.ZipFile(zip_path, 'r') as archive:
    readme = archive.read('README 2024.txt').decode('utf-8', errors='replace')

print(readme)

This brain tumor dataset containing 3064 T1-weighted contrast-inhanced images from 233 patients with three kinds of brain tumor: meningioma (708 slices), glioma (1426 slices), and pituitary tumor (930 slices). Due to the file size limit of repository, we split the whole dataset into 4 subsets, and achive them in 4 .zip files with each .zip file containing 766 slices.The 5-fold cross-validation indices are also provided (cvind.mat).

1. Data structure
This data is organized in matlab data format (.mat file). Each file stores a struct containing the following fields for an image:

cjdata.label: 1 for meningioma, 2 for glioma, 3 for pituitary tumor
cjdata.PID: patient ID
cjdata.image: image data
cjdata.tumorBorder: a vector storing the coordinates of discrete points on tumor border. For example, in [x1, y1, x2, y2,...], (xi, yi) are planar coordinates on the tumor border. They were generated by manually delineating the tumor border. So we can use it to generate binary image of tumor mask.

In [12]:
import io

In [13]:
import zipfile

with zipfile.ZipFile(zip_path, 'r') as archive:

    nested_zips = [
        name for name in archive.namelist()
        if name.endswith('.zip')
    ]

    for nested_zip in nested_zips:
        print("\n" + "="*60)
        print(nested_zip)

        nested_data = archive.read(nested_zip)

        with zipfile.ZipFile(io.BytesIO(nested_data), 'r') as inner:
            inner_files = inner.namelist()

            print("Number of files:", len(inner_files))
            print("First 5 files:")

            for file in inner_files[:5]:
                print("  ", file)


brainTumorDataPublic_1-766.zip
Number of files: 766
First 5 files:
   1.mat
   10.mat
   100.mat
   101.mat
   102.mat

brainTumorDataPublic_1533-2298.zip
Number of files: 766
First 5 files:
   1533.mat
   1534.mat
   1535.mat
   1536.mat
   1537.mat

brainTumorDataPublic_767-1532.zip
Number of files: 766
First 5 files:
   1000.mat
   1001.mat
   1002.mat
   1003.mat
   1004.mat

brainTumorDataPublic_2299-3064.zip
Number of files: 766
First 5 files:
   2299.mat
   2300.mat
   2301.mat
   2302.mat
   2303.mat


In [15]:
# import scipy.io

# with zipfile.ZipFile(zip_path, 'r') as archive:
#     cvind_data = archive.read('cvind.mat')

# cvind = scipy.io.loadmat(io.BytesIO(cvind_data))

# print("Variables in cvind.mat:")

# for key, value in cvind.items():
#     if not key.startswith('__'):
#         print(key, value.shape, value.dtype)
import h5py
import numpy as np
import io

# Reading cvind.mat from the original ZIP
with zipfile.ZipFile(zip_path, 'r') as archive:
    cvind_bytes = archive.read('cvind.mat')

# Saving a temporary copy in Colab
with open('/content/cvind.mat', 'wb') as f:
    f.write(cvind_bytes)

# Inspect the MATLAB v7.3 file
with h5py.File('/content/cvind.mat', 'r') as f:

    print("Variables:", list(f.keys()))

    for key in f.keys():
        data = f[key]

        print(f"\nVariable: {key}")
        print("Shape:", data.shape)
        print("Data type:", data.dtype)

        if isinstance(data, h5py.Dataset):
            values = data[()]

            if np.issubdtype(values.dtype, np.number):
                print("First 20 values:", values.reshape(-1)[:20])
                print("Unique values:", np.unique(values))

Variables: ['cvind']

Variable: cvind
Shape: (1, 3064)
Data type: float64
First 20 values: [5. 5. 5. 5. 5. 3. 3. 3. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 5.]
Unique values: [1. 2. 3. 4. 5.]


In [16]:
# Read the first nested ZIP from the original dataset
with zipfile.ZipFile(zip_path, 'r') as archive:
    nested_bytes = archive.read('brainTumorDataPublic_1-766.zip')

# Open the nested ZIP and read 1.mat
with zipfile.ZipFile(io.BytesIO(nested_bytes), 'r') as inner:
    mat_bytes = inner.read('1.mat')

# Save a temporary copy
with open('/content/1.mat', 'wb') as f:
    f.write(mat_bytes)

# Inspect its structure
with h5py.File('/content/1.mat', 'r') as f:

    print("Top-level variables:", list(f.keys()))

    def inspect_structure(name, obj):
        if isinstance(obj, h5py.Group):
            print(f"{name}: Group")
        else:
            print(f"{name}: Dataset | Shape: {obj.shape} | Type: {obj.dtype}")

    f.visititems(inspect_structure)

Top-level variables: ['cjdata']
cjdata: Group
cjdata/PID: Dataset | Shape: (6, 1) | Type: uint16
cjdata/image: Dataset | Shape: (512, 512) | Type: int16
cjdata/label: Dataset | Shape: (1, 1) | Type: float64
cjdata/tumorBorder: Dataset | Shape: (1, 38) | Type: float64
cjdata/tumorMask: Dataset | Shape: (512, 512) | Type: uint8
